In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 1


### Load Player Data and Bookmaker Data

In [4]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_41711/1373275659.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


### Update projected starting lineups

In [5]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Applications/Documents/NBA-Prop-Predictor/MODELS/teamInfo.py
Updated 14 teams with confirmed lineups


### Top EVs for single bets

In [7]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 250) & (usData['ODDS'] >= -250)]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=10, 
                             variance_inflation=1.1, distribution_type='t', 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']]
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Domantas Sabonis,Bovada,13.5,9.88,Under,190,1,11.96,119.6,0.630,High
1,Cedric Coward,Bovada,11.5,8.11,Under,175,1,10.83,108.3,0.619,Med
2,Domantas Sabonis,Bovada,14.5,9.88,Under,145,1,9.80,98.0,0.676,High
3,Jarace Walker,Bovada,10.5,7.11,Under,165,1,9.70,97.0,0.588,High
4,Anfernee Simons,Bovada,11.5,8.92,Under,175,1,8.81,88.1,0.503,High


## Top EVs for 2 leg bets

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Cedric Coward,Domantas Sabonis,13.5,16.5,under,under,1,10.39,0.520,Med,High
1,Jarace Walker,Domantas Sabonis,12.5,16.5,under,under,1,10.06,0.503,High,High
2,Anfernee Simons,Domantas Sabonis,14.5,16.5,under,under,1,9.90,0.495,High,High
3,Cedric Coward,Jarace Walker,13.5,12.5,under,under,1,9.37,0.468,Med,High
4,Anfernee Simons,Cedric Coward,14.5,13.5,under,under,1,9.21,0.461,High,Med


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Cedric Coward,Domantas Sabonis,13.5,16.5,under,under,1,10.39,0.520,Med,High
1,Jarace Walker,Domantas Sabonis,12.5,16.5,under,under,1,10.06,0.503,High,High
2,Cedric Coward,Jarace Walker,13.5,12.5,under,under,1,9.37,0.468,Med,High
3,Duncan Robinson,Domantas Sabonis,11.5,16.5,under,under,1,9.16,0.458,Med,High
4,Santi Aldama,Domantas Sabonis,12.5,16.5,under,under,1,9.10,0.455,High,High


## 3 leg parlay

### Underdog picks

In [11]:


dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=10, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Cedric Coward,Jarace Walker,Domantas Sabonis,13.5,12.5,16.5,under,under,under,1,20.83,0.417,Med,High,High
1,Anfernee Simons,Cedric Coward,Domantas Sabonis,14.5,13.5,16.5,under,under,under,1,20.59,0.412,High,Med,High
2,Anfernee Simons,Jarace Walker,Domantas Sabonis,14.5,12.5,16.5,under,under,under,1,20.10,0.402,High,High,High
3,Santi Aldama,Cedric Coward,Domantas Sabonis,12.5,13.5,16.5,under,under,under,1,19.36,0.387,High,Med,High
4,Tari Eason,Cedric Coward,Domantas Sabonis,11.5,13.5,16.5,under,under,under,1,19.32,0.386,High,Med,High


### Prizepicks picks

In [13]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Cedric Coward,Jarace Walker,Domantas Sabonis,13.5,12.5,16.5,under,under,under,1,20.83,0.417,Med,High,High
1,Anfernee Simons,Cedric Coward,Domantas Sabonis,14.5,13.5,16.5,under,under,under,1,20.59,0.412,High,Med,High
2,Anfernee Simons,Jarace Walker,Domantas Sabonis,14.5,12.5,16.5,under,under,under,1,20.10,0.402,High,High,High
3,Santi Aldama,Cedric Coward,Domantas Sabonis,12.5,13.5,16.5,under,under,under,1,19.36,0.387,High,Med,High
4,Tari Eason,Cedric Coward,Domantas Sabonis,11.5,13.5,16.5,under,under,under,1,19.32,0.386,High,Med,High


In [14]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
174,PrizePicks,player_points,Giannis Antetokounmpo,Over,30.0,-137,2025-11-09,2025-11-09T15:35:45Z
176,PrizePicks,player_points,Kevin Durant,Over,25.5,-137,2025-11-09,2025-11-09T15:35:45Z
178,PrizePicks,player_points,Alperen Sengun,Over,21.5,-137,2025-11-09,2025-11-09T15:35:45Z
180,PrizePicks,player_points,Amen Thompson,Over,16.5,-137,2025-11-09,2025-11-09T15:35:45Z
182,PrizePicks,player_points,Jabari Smith Jr,Over,13.5,-137,2025-11-09,2025-11-09T15:35:45Z
...,...,...,...,...,...,...,...,...
2657,PrizePicks,player_blocks_steals,Jimmy Butler,Over,1.5,-137,2025-11-10,2025-11-09T15:35:44Z
2659,PrizePicks,player_blocks_steals,Pascal Siakam,Over,1.5,-137,2025-11-10,2025-11-09T15:35:44Z
2661,PrizePicks,player_blocks_steals,Anthony Edwards,Over,1.5,-137,2025-11-10,2025-11-09T15:35:41Z
2663,PrizePicks,player_blocks_steals,Naz Reid,Over,1.5,-137,2025-11-10,2025-11-09T15:35:41Z


In [15]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 83 records for player_points to player_points.csv
Saved 57 records for player_rebounds to player_rebounds.csv
Saved 37 records for player_assists to player_assists.csv
Saved 14 records for player_threes to player_threes.csv
Saved 5 records for player_blocks to player_blocks.csv
Saved 10 records for player_steals to player_steals.csv
Saved 27 records for player_field_goals to player_field_goals.csv
Saved 16 records for player_frees_made to player_frees_made.csv
Saved 85 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 83 records for player_points_rebounds to player_points_rebounds.csv
Saved 74 records for player_points_assists to player_points_assists.csv
Saved 55 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 7 records for player_turnovers to player_turnovers.csv
Saved 9 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS


In [16]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 75 records for player_points to player_points.csv
Saved 27 records for player_rebounds to player_rebounds.csv
Saved 19 records for player_assists to player_assists.csv
Saved 13 records for player_threes to player_threes.csv
Saved 2 records for player_blocks to player_blocks.csv
Saved 1 records for player_steals to player_steals.csv
Saved 80 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 38 records for player_points_rebounds to player_points_rebounds.csv
Saved 29 records for player_points_assists to player_points_assists.csv
Saved 24 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 4 records for player_turnovers to player_turnovers.csv
Saved 3 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
